In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Execution Analysis Module Initialized")

## 1. Generate Intraday Market Data

In [ ]:
# Generate high-frequency price and volume data
np.random.seed(42)

# Trading day: 9:30 AM - 4:00 PM (390 minutes)
n_minutes = 390
timestamps = pd.date_range(start='2024-01-15 09:30:00', periods=n_minutes, freq='1min')

# Price process with intraday patterns
initial_price = 100.0
drift = 0.00001  # Per-minute drift
volatility = 0.001  # Per-minute volatility

# Generate realistic intraday volume pattern (U-shaped)
time_of_day = np.arange(n_minutes) / n_minutes
volume_pattern = 1.5 * (time_of_day**2 - time_of_day + 0.5)  # U-shaped curve
base_volume = 10000
volume = (base_volume * volume_pattern * (1 + np.random.randn(n_minutes) * 0.2)).astype(int)
volume = np.maximum(volume, 1000)  # Minimum volume

# Price with mean reversion
returns = np.random.normal(drift, volatility, n_minutes)
# Add mean reversion component
mean_reversion_speed = 0.01
for i in range(1, n_minutes):
    deviation = returns[:i].sum()
    returns[i] -= mean_reversion_speed * deviation

prices = initial_price * np.exp(np.cumsum(returns))

# Generate bid-ask spread
spread_bps = 5  # 5 basis points
spread = prices * (spread_bps / 10000)
bid_prices = prices - spread / 2
ask_prices = prices + spread / 2

# Create DataFrame
market_data = pd.DataFrame({
    'timestamp': timestamps,
    'price': prices,
    'bid': bid_prices,
    'ask': ask_prices,
    'volume': volume,
    'spread': spread
})
market_data.set_index('timestamp', inplace=True)

# Calculate VWAP
market_data['vwap'] = (market_data['price'] * market_data['volume']).cumsum() / market_data['volume'].cumsum()

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price and spread
axes[0].plot(market_data.index, market_data['price'], 'b-', linewidth=1.5, label='Mid Price')
axes[0].fill_between(market_data.index, market_data['bid'], market_data['ask'], 
                     alpha=0.3, color='gray', label='Bid-Ask Spread')
axes[0].plot(market_data.index, market_data['vwap'], 'r--', linewidth=2, label='VWAP')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Intraday Price Evolution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume profile
axes[1].bar(market_data.index, market_data['volume'], width=0.0007, alpha=0.7, color='orange')
axes[1].set_ylabel('Volume')
axes[1].set_title('Intraday Volume Pattern (U-shaped)')
axes[1].grid(True, alpha=0.3)

# Spread in basis points
spread_bps_actual = (market_data['spread'] / market_data['price']) * 10000
axes[2].plot(market_data.index, spread_bps_actual, 'g-', linewidth=1, alpha=0.7)
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Spread (bps)')
axes[2].set_title('Bid-Ask Spread')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Trading session: {market_data.index[0]} to {market_data.index[-1]}")
print(f"Opening price: ${market_data['price'].iloc[0]:.2f}")
print(f"Closing price: ${market_data['price'].iloc[-1]:.2f}")
print(f"Day VWAP: ${market_data['vwap'].iloc[-1]:.2f}")
print(f"Total volume: {market_data['volume'].sum():,}")
print(f"Average spread: {spread_bps_actual.mean():.2f} bps")

## 2. TWAP (Time-Weighted Average Price) Algorithm

In [ ]:
@dataclass
class ExecutionResult:
    timestamps: List[datetime]
    prices: List[float]
    quantities: List[int]
    slippage: List[float]
    
    @property
    def average_price(self):
        total_notional = sum(p * q for p, q in zip(self.prices, self.quantities))
        total_quantity = sum(self.quantities)
        return total_notional / total_quantity if total_quantity > 0 else 0
    
    @property
    def total_slippage(self):
        return sum(self.slippage)

def execute_twap(market_data: pd.DataFrame, total_shares: int, n_slices: int = 10) -> ExecutionResult:
    """Execute TWAP strategy - evenly distribute orders over time"""
    execution_times = np.linspace(0, len(market_data)-1, n_slices, dtype=int)
    shares_per_slice = total_shares // n_slices
    
    timestamps = []
    prices = []
    quantities = []
    slippage = []
    
    for i, time_idx in enumerate(execution_times):
        timestamp = market_data.index[time_idx]
        
        # Execute at ask price (market order)
        execution_price = market_data['ask'].iloc[time_idx]
        mid_price = market_data['price'].iloc[time_idx]
        
        # Calculate market impact (square root law)
        market_volume = market_data['volume'].iloc[time_idx]
        participation_rate = shares_per_slice / market_volume
        impact = mid_price * 0.1 * np.sqrt(participation_rate)  # Simplified impact model
        
        # Total execution cost
        total_price = execution_price + impact
        slip = total_price - mid_price
        
        timestamps.append(timestamp)
        prices.append(total_price)
        quantities.append(shares_per_slice)
        slippage.append(slip * shares_per_slice)
    
    return ExecutionResult(timestamps, prices, quantities, slippage)

# Execute TWAP
total_order_size = 100000  # shares
twap_result = execute_twap(market_data, total_order_size, n_slices=10)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Execution points on price chart
axes[0].plot(market_data.index, market_data['price'], 'gray', alpha=0.5, linewidth=1, label='Market Price')
axes[0].plot(market_data.index, market_data['vwap'], 'b--', linewidth=2, label='VWAP')
axes[0].scatter(twap_result.timestamps, twap_result.prices, color='red', s=150, 
               marker='o', zorder=5, label='TWAP Executions', edgecolors='black', linewidths=2)
axes[0].axhline(y=twap_result.average_price, color='red', linestyle='--', 
               linewidth=2, alpha=0.7, label=f'TWAP Avg: ${twap_result.average_price:.2f}')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('TWAP Execution Strategy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative quantity and slippage
cumulative_qty = np.cumsum(twap_result.quantities)
cumulative_slippage = np.cumsum(twap_result.slippage)

ax2 = axes[1]
ax2.bar(range(len(twap_result.quantities)), twap_result.quantities, 
       alpha=0.7, color='blue', label='Quantity per Slice')
ax2.set_xlabel('Execution Slice')
ax2.set_ylabel('Shares', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

ax2_twin = ax2.twinx()
ax2_twin.plot(range(len(cumulative_slippage)), cumulative_slippage, 
             'ro-', linewidth=2, markersize=8, label='Cumulative Slippage')
ax2_twin.set_ylabel('Cumulative Slippage ($)', color='red')
ax2_twin.tick_params(axis='y', labelcolor='red')

axes[1].set_title('TWAP Execution Schedule & Costs')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance metrics
arrival_price = market_data['price'].iloc[0]
final_vwap = market_data['vwap'].iloc[-1]

print("\n=== TWAP Execution Results ===")
print(f"Total shares executed: {sum(twap_result.quantities):,}")
print(f"Number of slices: {len(twap_result.quantities)}")
print(f"Average execution price: ${twap_result.average_price:.4f}")
print(f"Arrival price: ${arrival_price:.4f}")
print(f"Day VWAP: ${final_vwap:.4f}")
print(f"\nPerformance vs arrival: {(twap_result.average_price - arrival_price):.4f} (${twap_result.total_slippage:.2f})")
print(f"Performance vs VWAP: {(twap_result.average_price - final_vwap):.4f}")
print(f"Slippage (bps): {(twap_result.total_slippage / (arrival_price * total_order_size)) * 10000:.2f}")

## 3. VWAP (Volume-Weighted Average Price) Algorithm

In [ ]:
def execute_vwap(market_data: pd.DataFrame, total_shares: int, participation_rate: float = 0.1) -> ExecutionResult:
    """Execute VWAP strategy - follow volume profile"""
    # Calculate target volume profile
    total_market_volume = market_data['volume'].sum()
    volume_distribution = market_data['volume'] / total_market_volume
    target_shares = (volume_distribution * total_shares).round().astype(int)
    
    # Adjust for rounding
    diff = total_shares - target_shares.sum()
    target_shares.iloc[-1] += diff
    
    timestamps = []
    prices = []
    quantities = []
    slippage = []
    
    for idx, (timestamp, row) in enumerate(market_data.iterrows()):
        shares = target_shares.iloc[idx]
        
        if shares > 0:
            # Execute at ask price
            execution_price = row['ask']
            mid_price = row['price']
            
            # Market impact based on participation rate
            actual_participation = shares / row['volume']
            impact = mid_price * 0.1 * np.sqrt(actual_participation)
            
            total_price = execution_price + impact
            slip = total_price - mid_price
            
            timestamps.append(timestamp)
            prices.append(total_price)
            quantities.append(shares)
            slippage.append(slip * shares)
    
    return ExecutionResult(timestamps, prices, quantities, slippage)

# Execute VWAP
vwap_result = execute_vwap(market_data, total_order_size, participation_rate=0.1)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Execution points
axes[0].plot(market_data.index, market_data['price'], 'gray', alpha=0.5, linewidth=1, label='Market Price')
axes[0].plot(market_data.index, market_data['vwap'], 'b--', linewidth=2, label='Market VWAP')
axes[0].scatter(vwap_result.timestamps, vwap_result.prices, color='green', s=20, 
               alpha=0.6, zorder=5, label='VWAP Executions')
axes[0].axhline(y=vwap_result.average_price, color='green', linestyle='--', 
               linewidth=2, alpha=0.7, label=f'Execution Avg: ${vwap_result.average_price:.2f}')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('VWAP Execution Strategy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume comparison
# Group VWAP executions by minute for visualization
vwap_volume_df = pd.DataFrame({
    'timestamp': vwap_result.timestamps,
    'quantity': vwap_result.quantities
}).set_index('timestamp')

axes[1].bar(market_data.index, market_data['volume'], width=0.0007, 
           alpha=0.4, color='gray', label='Market Volume')
axes[1].bar(vwap_volume_df.index, vwap_volume_df['quantity'], width=0.0007, 
           alpha=0.7, color='green', label='Execution Volume')
axes[1].set_ylabel('Volume')
axes[1].set_title('Volume Profile - Market vs Execution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Cumulative execution
cumulative_exec = np.cumsum(vwap_result.quantities)
axes[2].plot(range(len(cumulative_exec)), cumulative_exec, 'g-', linewidth=2)
axes[2].axhline(y=total_order_size, color='red', linestyle='--', alpha=0.5, label='Target')
axes[2].set_xlabel('Execution Number')
axes[2].set_ylabel('Cumulative Shares')
axes[2].set_title('Cumulative Execution Progress')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== VWAP Execution Results ===")
print(f"Total shares executed: {sum(vwap_result.quantities):,}")
print(f"Number of executions: {len(vwap_result.quantities)}")
print(f"Average execution price: ${vwap_result.average_price:.4f}")
print(f"Arrival price: ${arrival_price:.4f}")
print(f"Day VWAP: ${final_vwap:.4f}")
print(f"\nPerformance vs arrival: {(vwap_result.average_price - arrival_price):.4f} (${vwap_result.total_slippage:.2f})")
print(f"Performance vs VWAP: {(vwap_result.average_price - final_vwap):.4f}")
print(f"Slippage (bps): {(vwap_result.total_slippage / (arrival_price * total_order_size)) * 10000:.2f}")

## 4. Implementation Shortfall Analysis

In [ ]:
def calculate_implementation_shortfall(execution_result: ExecutionResult, 
                                      arrival_price: float,
                                      decision_price: float,
                                      total_shares: int) -> dict:
    """Calculate implementation shortfall components"""
    # Benchmark: arrival price
    benchmark_cost = arrival_price * total_shares
    
    # Actual execution cost
    actual_cost = sum(p * q for p, q in zip(execution_result.prices, execution_result.quantities))
    
    # Total implementation shortfall
    total_is = actual_cost - benchmark_cost
    
    # Decomposition
    # 1. Delay cost (decision to arrival)
    delay_cost = (arrival_price - decision_price) * total_shares
    
    # 2. Market impact + timing cost
    execution_cost = total_is - delay_cost
    
    return {
        'total_is': total_is,
        'total_is_bps': (total_is / benchmark_cost) * 10000,
        'delay_cost': delay_cost,
        'delay_cost_bps': (delay_cost / benchmark_cost) * 10000,
        'execution_cost': execution_cost,
        'execution_cost_bps': (execution_cost / benchmark_cost) * 10000,
        'benchmark_cost': benchmark_cost,
        'actual_cost': actual_cost
    }

# Calculate IS for both strategies
decision_price = arrival_price * 0.999  # Decision made slightly before market open

is_twap = calculate_implementation_shortfall(twap_result, arrival_price, decision_price, total_order_size)
is_vwap = calculate_implementation_shortfall(vwap_result, arrival_price, decision_price, total_order_size)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# IS components comparison - TWAP
components_twap = ['Delay Cost', 'Execution Cost', 'Total IS']
values_twap = [is_twap['delay_cost_bps'], is_twap['execution_cost_bps'], is_twap['total_is_bps']]
colors_twap = ['orange', 'red', 'darkred']

axes[0, 0].bar(components_twap, values_twap, color=colors_twap, alpha=0.7, edgecolor='black')
axes[0, 0].set_ylabel('Cost (bps)')
axes[0, 0].set_title('TWAP - Implementation Shortfall Breakdown')
axes[0, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values_twap):
    axes[0, 0].text(i, v, f'{v:.1f}', ha='center', va='bottom')

# IS components comparison - VWAP
components_vwap = ['Delay Cost', 'Execution Cost', 'Total IS']
values_vwap = [is_vwap['delay_cost_bps'], is_vwap['execution_cost_bps'], is_vwap['total_is_bps']]
colors_vwap = ['orange', 'green', 'darkgreen']

axes[0, 1].bar(components_vwap, values_vwap, color=colors_vwap, alpha=0.7, edgecolor='black')
axes[0, 1].set_ylabel('Cost (bps)')
axes[0, 1].set_title('VWAP - Implementation Shortfall Breakdown')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values_vwap):
    axes[0, 1].text(i, v, f'{v:.1f}', ha='center', va='bottom')

# Strategy comparison
strategies = ['TWAP', 'VWAP']
total_is_values = [is_twap['total_is_bps'], is_vwap['total_is_bps']]
execution_costs = [is_twap['execution_cost_bps'], is_vwap['execution_cost_bps']]

x = np.arange(len(strategies))
width = 0.35

axes[1, 0].bar(x - width/2, total_is_values, width, label='Total IS', alpha=0.8)
axes[1, 0].bar(x + width/2, execution_costs, width, label='Execution Cost', alpha=0.8)
axes[1, 0].set_ylabel('Cost (bps)')
axes[1, 0].set_title('Strategy Comparison')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(strategies)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Cost breakdown pie chart
avg_delay = (is_twap['delay_cost_bps'] + is_vwap['delay_cost_bps']) / 2
avg_exec_twap = is_twap['execution_cost_bps']
avg_exec_vwap = is_vwap['execution_cost_bps']

sizes = [abs(avg_delay), abs(avg_exec_twap), abs(avg_exec_vwap)]
labels = ['Delay Cost', 'TWAP Execution', 'VWAP Execution']
colors_pie = ['orange', 'red', 'green']

axes[1, 1].pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', 
              startangle=90, textprops={'fontsize': 10})
axes[1, 1].set_title('Average Cost Distribution')

plt.tight_layout()
plt.show()

print("\n=== Implementation Shortfall Analysis ===")
print("\nTWAP:")
for key, value in is_twap.items():
    if 'bps' in key:
        print(f"  {key}: {value:.2f} bps")
    else:
        print(f"  {key}: ${value:.2f}")

print("\nVWAP:")
for key, value in is_vwap.items():
    if 'bps' in key:
        print(f"  {key}: {value:.2f} bps")
    else:
        print(f"  {key}: ${value:.2f}")

## 5. Market Impact Model

In [ ]:
def almgren_chriss_impact(order_size: int, market_volume: float, volatility: float, 
                         price: float, permanent_factor: float = 0.1, 
                         temporary_factor: float = 0.01) -> Tuple[float, float]:
    """Calculate market impact using Almgren-Chriss model"""
    # Participation rate
    participation = order_size / market_volume
    
    # Permanent impact (price moves and doesn't revert)
    permanent_impact = price * permanent_factor * volatility * participation
    
    # Temporary impact (price reverts after trade)
    temporary_impact = price * temporary_factor * volatility * np.sqrt(participation)
    
    return permanent_impact, temporary_impact

# Analyze impact for different order sizes
order_sizes = np.linspace(1000, 200000, 50)
daily_volume = market_data['volume'].sum()
avg_volatility = market_data['price'].pct_change().std()
avg_price = market_data['price'].mean()

permanent_impacts = []
temporary_impacts = []
total_impacts = []

for size in order_sizes:
    perm, temp = almgren_chriss_impact(size, daily_volume, avg_volatility, avg_price)
    permanent_impacts.append(perm)
    temporary_impacts.append(temp)
    total_impacts.append(perm + temp)

# Convert to basis points
permanent_impacts_bps = np.array(permanent_impacts) / avg_price * 10000
temporary_impacts_bps = np.array(temporary_impacts) / avg_price * 10000
total_impacts_bps = np.array(total_impacts) / avg_price * 10000

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Impact vs order size
axes[0, 0].plot(order_sizes, total_impacts_bps, 'b-', linewidth=2, label='Total Impact')
axes[0, 0].plot(order_sizes, permanent_impacts_bps, 'r--', linewidth=2, label='Permanent')
axes[0, 0].plot(order_sizes, temporary_impacts_bps, 'g--', linewidth=2, label='Temporary')
axes[0, 0].axvline(x=total_order_size, color='black', linestyle=':', alpha=0.5, label='Our Order')
axes[0, 0].set_xlabel('Order Size (shares)')
axes[0, 0].set_ylabel('Market Impact (bps)')
axes[0, 0].set_title('Market Impact vs Order Size')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Impact vs participation rate
participation_rates = order_sizes / daily_volume * 100
axes[0, 1].plot(participation_rates, total_impacts_bps, 'b-', linewidth=2)
axes[0, 1].axvline(x=total_order_size/daily_volume*100, color='red', linestyle='--', 
                  alpha=0.7, label=f'Our Order: {total_order_size/daily_volume*100:.1f}%')
axes[0, 1].set_xlabel('Participation Rate (%)')
axes[0, 1].set_ylabel('Total Market Impact (bps)')
axes[0, 1].set_title('Market Impact vs Participation Rate')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Impact decomposition for our order
our_perm, our_temp = almgren_chriss_impact(total_order_size, daily_volume, avg_volatility, avg_price)
impact_components = ['Permanent', 'Temporary']
impact_values = [our_perm / avg_price * 10000, our_temp / avg_price * 10000]

axes[1, 0].bar(impact_components, impact_values, color=['red', 'green'], alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel('Impact (bps)')
axes[1, 0].set_title(f'Impact Breakdown for {total_order_size:,} Share Order')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(impact_values):
    axes[1, 0].text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Optimal execution time (trade-off between impact and risk)
execution_times = np.linspace(1, 390, 50)  # minutes
slices = 390 / execution_times
impact_per_slice = total_impacts_bps[np.argmin(np.abs(order_sizes - total_order_size))] / slices
timing_risk = avg_volatility * np.sqrt(execution_times) * 10000  # Simplified

axes[1, 1].plot(execution_times, impact_per_slice * slices, 'r-', linewidth=2, label='Total Impact')
axes[1, 1].plot(execution_times, timing_risk, 'b-', linewidth=2, label='Timing Risk')
axes[1, 1].plot(execution_times, impact_per_slice * slices + timing_risk, 'g-', 
               linewidth=2.5, label='Total Cost')
axes[1, 1].set_xlabel('Execution Time (minutes)')
axes[1, 1].set_ylabel('Cost (bps)')
axes[1, 1].set_title('Optimal Execution Time Analysis')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Market Impact Analysis ===")
print(f"Order size: {total_order_size:,} shares")
print(f"Daily volume: {daily_volume:,} shares")
print(f"Participation rate: {total_order_size/daily_volume*100:.2f}%")
print(f"\nPermanent impact: {our_perm/avg_price*10000:.2f} bps")
print(f"Temporary impact: {our_temp/avg_price*10000:.2f} bps")
print(f"Total impact: {(our_perm+our_temp)/avg_price*10000:.2f} bps")
print(f"\nCost in dollars: ${our_perm + our_temp:.2f}")

## 6. Summary

### Key Concepts:

1. **TWAP (Time-Weighted Average Price)**
   - Evenly distributes trades over time
   - Minimizes timing risk
   - Good for less liquid markets

2. **VWAP (Volume-Weighted Average Price)**
   - Follows market volume profile
   - Minimizes market impact
   - Standard benchmark for execution

3. **Implementation Shortfall**
   - Measures total execution cost
   - Decomposes into delay and execution costs
   - Important performance metric

4. **Market Impact**
   - Permanent vs temporary impact
   - Square root law for participation
   - Trade-off between speed and cost

5. **Execution Quality Metrics**
   - Slippage analysis
   - Benchmark comparisons
   - Cost attribution

### Best Practices:
- Choose algorithm based on urgency and market conditions
- Monitor participation rates
- Track execution vs benchmarks
- Account for market impact in strategy design
- Optimize execution schedule for cost-risk trade-off